# Regression Track

## Linear Regression

In [2]:
%%capture
%run regression_preprocessing.ipynb

In [3]:
#Imports

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
# Common function for evaluating every regression model

def evaluate_regression_model(model, X_test_data, y_test_data):
    predictions = model.predict(X_test_data)

    r2 = r2_score(y_test_data, predictions)
    rmse = np.sqrt(mean_squared_error(y_test_data, predictions))
    mae = mean_absolute_error(y_test_data, predictions)

    return r2, rmse, mae, predictions

regression_results = []
model_registry = {}  # name -> dict(estimator, X_train, X_test, predictions)

## Linear Regression

In [5]:
# Linear Regression - Baseline

linear_model = LinearRegression()

linear_model.fit(X_train_encoded, y_train)

linear_r2, linear_rmse, linear_mae, linear_predictions = evaluate_regression_model(
    linear_model,
    X_test_encoded,
    y_test
)

print("Linear Regression — Baseline")
print(f"R²   : {linear_r2:.4f}")
print(f"RMSE : {linear_rmse:.4f}")
print(f"MAE  : {linear_mae:.4f}")

# Save baseline results
linear_baseline_r2 = linear_r2
linear_baseline_rmse = linear_rmse
linear_baseline_mae = linear_mae


Linear Regression — Baseline
R²   : 0.4343
RMSE : 5.6825
MAE  : 2.7586


In [6]:
# Linear Regression - Hyperparameter Tuning
# 3-fold CV is used to reduce repeated fitting time.

linear_param_grid = {
    'fit_intercept': [True, False]
}

linear_grid = GridSearchCV(
    estimator=LinearRegression(),
    param_grid=linear_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

linear_grid.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(linear_grid.best_params_)
print(f"\nBest Cross-Validation R²: {linear_grid.best_score_:.4f}")


Best Parameters:
{'fit_intercept': False}

Best Cross-Validation R²: 0.4351


In [7]:
# Evaluating Tuned Linear Regression

linear_best = linear_grid.best_estimator_

linear_r2, linear_rmse, linear_mae, linear_predictions = evaluate_regression_model(
    linear_best,
    X_test_encoded,
    y_test
)

print("Tuned Linear Regression")
print(f"R²   : {linear_r2:.4f}")
print(f"RMSE : {linear_rmse:.4f}")
print(f"MAE  : {linear_mae:.4f}")

# Save tuned results
linear_tuned_r2 = linear_r2
linear_tuned_rmse = linear_rmse
linear_tuned_mae = linear_mae


Tuned Linear Regression
R²   : 0.4343
RMSE : 5.6825
MAE  : 2.7586


In [8]:
# Linear Regression - Baseline vs Tuned Comparison

linear_comparison = pd.DataFrame({
    'Model': ['Baseline Linear Regression', 'Tuned Linear Regression'],
    'R²': [linear_baseline_r2, linear_tuned_r2],
    'RMSE': [linear_baseline_rmse, linear_tuned_rmse],
    'MAE': [linear_baseline_mae, linear_tuned_mae]
})

display(linear_comparison)


,Model,R²,RMSE,MAE
0,Baseline Linear Regression,0.434277,5.682515,2.758578
1,Tuned Linear Regression,0.434277,5.682517,2.758556


In [9]:
# Linear Regression - Coefficients

linear_coefficients = pd.DataFrame({
    'Feature': all_feature_cols,
    'Coefficient': linear_best.coef_
}).sort_values(
    by='Coefficient',
    key=np.abs,
    ascending=False
)

print("Linear Regression Coefficients:")
display(linear_coefficients)


Linear Regression Coefficients:


,Feature,Coefficient
221,categorical__Facility Name_Rockefeller Univers...,27.867391
82,categorical__Facility Name_Blythedale Children...,24.756154
104,categorical__Facility Name_Coler-Goldwater Spe...,24.150759
103,categorical__Facility Name_Coler-Goldwater Spe...,22.609316
1602,categorical__APR DRG Code_588,21.459739
...,...,...
330,categorical__Zip Code - 3 digits_123,-0.002433
1529,categorical__APR DRG Code_344,-0.002379
1898,categorical__APR DRG Description_OSTEOMYELITIS...,-0.002379
74,categorical__Facility Name_Arnot Ogden Medical...,0.000879


In [10]:
# Adding Linear Regression to regression results

regression_results.append({
    'Model': 'Linear Regression',
    'R²': linear_tuned_r2,
    'RMSE': linear_tuned_rmse,
    'MAE': linear_tuned_mae
})

model_registry['Linear Regression'] = {
    'estimator': linear_best,
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'predictions': linear_predictions
}


## Ridge Regression

In [11]:
# Ridge Regression - Baseline

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_encoded, y_train)

ridge_r2, ridge_rmse, ridge_mae, ridge_predictions = evaluate_regression_model(
    ridge_model,
    X_test_encoded,
    y_test
)

print("Ridge Regression — Baseline")
print(f"R²   : {ridge_r2:.4f}")
print(f"RMSE : {ridge_rmse:.4f}")
print(f"MAE  : {ridge_mae:.4f}")

# Save baseline results
ridge_baseline_r2 = ridge_r2
ridge_baseline_rmse = ridge_rmse
ridge_baseline_mae = ridge_mae


Ridge Regression — Baseline
R²   : 0.4343
RMSE : 5.6825
MAE  : 2.7586


In [12]:
# Ridge Regression - Hyperparameter Tuning
# 3-fold CV reduces the number of model fits.

ridge_param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_grid = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

ridge_grid.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(ridge_grid.best_params_)
print(f"\nBest Cross-Validation R²: {ridge_grid.best_score_:.4f}")


Best Parameters:
{'alpha': 10.0}

Best Cross-Validation R²: 0.4352


In [13]:
# Evaluating Tuned Ridge Regression

ridge_best = ridge_grid.best_estimator_

ridge_r2, ridge_rmse, ridge_mae, ridge_predictions = evaluate_regression_model(
    ridge_best,
    X_test_encoded,
    y_test
)

print("Tuned Ridge Regression")
print(f"R²   : {ridge_r2:.4f}")
print(f"RMSE : {ridge_rmse:.4f}")
print(f"MAE  : {ridge_mae:.4f}")

# Save tuned results
ridge_tuned_r2 = ridge_r2
ridge_tuned_rmse = ridge_rmse
ridge_tuned_mae = ridge_mae


Tuned Ridge Regression
R²   : 0.4343
RMSE : 5.6825
MAE  : 2.7587


In [14]:
# Ridge Regression - Baseline vs Tuned Comparison

ridge_comparison = pd.DataFrame({
    'Model': ['Baseline Ridge', 'Tuned Ridge'],
    'R²': [ridge_baseline_r2, ridge_tuned_r2],
    'RMSE': [ridge_baseline_rmse, ridge_tuned_rmse],
    'MAE': [ridge_baseline_mae, ridge_tuned_mae]
})

display(ridge_comparison)


,Model,R²,RMSE,MAE
0,Baseline Ridge,0.434289,5.682457,2.758552
1,Tuned Ridge,0.434276,5.682521,2.758668


In [15]:
# Ridge Regression - Coefficients

ridge_coefficients = pd.DataFrame({
    'Feature': all_feature_cols,
    'Coefficient': ridge_best.coef_
}).sort_values(
    by='Coefficient',
    key=np.abs,
    ascending=False
)

print("Ridge Regression Coefficients:")
display(ridge_coefficients)


Ridge Regression Coefficients:


,Feature,Coefficient
104,categorical__Facility Name_Coler-Goldwater Spe...,23.837677
82,categorical__Facility Name_Blythedale Children...,23.793756
103,categorical__Facility Name_Coler-Goldwater Spe...,22.385155
1602,categorical__APR DRG Code_588,20.234618
1881,categorical__APR DRG Description_NEONATE BWT <...,20.234618
...,...,...
967,categorical__CCS Procedure Code_47.0,-0.001056
1196,categorical__CCS Procedure Description_DX CARD...,-0.001056
948,categorical__CCS Procedure Code_27.0,0.000110
1182,categorical__CCS Procedure Description_CONTROL...,0.000110


In [16]:
# Adding Ridge Regression to regression results

regression_results.append({
    'Model': 'Ridge Regression',
    'R²': ridge_tuned_r2,
    'RMSE': ridge_tuned_rmse,
    'MAE': ridge_tuned_mae
})

model_registry['Ridge Regression'] = {
    'estimator': ridge_best,
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'predictions': ridge_predictions
}


## Lasso Regression

In [17]:
# Lasso Regression - Baseline

lasso_model = Lasso(alpha=0.1)

lasso_model.fit(X_train_encoded, y_train)

lasso_r2, lasso_rmse, lasso_mae, lasso_predictions = evaluate_regression_model(
    lasso_model,
    X_test_encoded,
    y_test
)

print("Lasso Regression — Baseline")
print(f"R²   : {lasso_r2:.4f}")
print(f"RMSE : {lasso_rmse:.4f}")
print(f"MAE  : {lasso_mae:.4f}")

# Save baseline results
lasso_baseline_r2 = lasso_r2
lasso_baseline_rmse = lasso_rmse
lasso_baseline_mae = lasso_mae


Lasso Regression — Baseline
R²   : 0.2342
RMSE : 6.6115
MAE  : 3.2421


In [18]:
# Lasso Regression - Hyperparameter Tuning
# 3-fold CV + a bounded iteration count keeps tuning faster.

lasso_param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
}

lasso_grid = GridSearchCV(
    estimator=Lasso(max_iter=5000),
    param_grid=lasso_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

lasso_grid.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(lasso_grid.best_params_)
print(f"\nBest Cross-Validation R²: {lasso_grid.best_score_:.4f}")


Best Parameters:
{'alpha': 0.001}

Best Cross-Validation R²: 0.4265


In [19]:
# Evaluating Tuned Lasso Regression

lasso_best = lasso_grid.best_estimator_

lasso_r2, lasso_rmse, lasso_mae, lasso_predictions = evaluate_regression_model(
    lasso_best,
    X_test_encoded,
    y_test
)

print("Tuned Lasso Regression")
print(f"R²   : {lasso_r2:.4f}")
print(f"RMSE : {lasso_rmse:.4f}")
print(f"MAE  : {lasso_mae:.4f}")

# Save tuned results
lasso_tuned_r2 = lasso_r2
lasso_tuned_rmse = lasso_rmse
lasso_tuned_mae = lasso_mae


Tuned Lasso Regression
R²   : 0.4248
RMSE : 5.7301
MAE  : 2.7801


In [20]:
# Lasso Regression - Baseline vs Tuned Comparison

lasso_comparison = pd.DataFrame({
    'Model': ['Baseline Lasso', 'Tuned Lasso'],
    'R²': [lasso_baseline_r2, lasso_tuned_r2],
    'RMSE': [lasso_baseline_rmse, lasso_tuned_rmse],
    'MAE': [lasso_baseline_mae, lasso_tuned_mae]
})

display(lasso_comparison)


,Model,R²,RMSE,MAE
0,Baseline Lasso,0.234184,6.611517,3.242059
1,Tuned Lasso,0.424772,5.730057,2.780058


In [21]:
# Lasso Regression - Coefficients

lasso_coefficients = pd.DataFrame({
    'Feature': all_feature_cols,
    'Coefficient': lasso_best.coef_
}).sort_values(
    by='Coefficient',
    key=np.abs,
    ascending=False
)

print("Lasso Regression Coefficients:")
display(lasso_coefficients)

print(f"\nNon-zero coefficients: {np.count_nonzero(lasso_best.coef_)} / {len(lasso_best.coef_)}")


Lasso Regression Coefficients:


,Feature,Coefficient
1605,categorical__APR DRG Code_593,36.152812
1602,categorical__APR DRG Code_588,35.398984
1606,categorical__APR DRG Code_602,30.843305
1380,categorical__APR DRG Code_4,27.588047
104,categorical__Facility Name_Coler-Goldwater Spe...,20.533263
...,...,...
856,categorical__CCS Diagnosis Description_PARKINS...,-0.000000
855,categorical__CCS Diagnosis Description_PARALYSIS,-0.000000
852,categorical__CCS Diagnosis Description_OVARIAN...,-0.000000
850,categorical__CCS Diagnosis Description_OTITIS ...,0.000000



Non-zero coefficients: 806 / 2111


In [22]:
# Adding Lasso Regression to regression results

regression_results.append({
    'Model': 'Lasso Regression',
    'R²': lasso_tuned_r2,
    'RMSE': lasso_tuned_rmse,
    'MAE': lasso_tuned_mae
})

model_registry['Lasso Regression'] = {
    'estimator': lasso_best,
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'predictions': lasso_predictions
}


## ElasticNet Regression

In [23]:
# ElasticNet Regression - Baseline

elastic_model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=10000
)

elastic_model.fit(X_train_encoded, y_train)

elastic_r2, elastic_rmse, elastic_mae, elastic_predictions = evaluate_regression_model(
    elastic_model,
    X_test_encoded,
    y_test
)

print("ElasticNet Regression — Baseline")
print(f"R²   : {elastic_r2:.4f}")
print(f"RMSE : {elastic_rmse:.4f}")
print(f"MAE  : {elastic_mae:.4f}")

# Save baseline results
elastic_baseline_r2 = elastic_r2
elastic_baseline_rmse = elastic_rmse
elastic_baseline_mae = elastic_mae


ElasticNet Regression — Baseline
R²   : 0.2449
RMSE : 6.5653
MAE  : 3.2000


In [24]:
# ElasticNet Regression - Hyperparameter Tuning
# RandomizedSearchCV avoids evaluating the full 25-combination grid.

elastic_param_distributions = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elastic_grid = RandomizedSearchCV(
    estimator=ElasticNet(max_iter=5000, random_state=42),
    param_distributions=elastic_param_distributions,
    n_iter=6,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

elastic_grid.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(elastic_grid.best_params_)
print(f"\nBest Cross-Validation R²: {elastic_grid.best_score_:.4f}")


Best Parameters:
{'l1_ratio': 0.1, 'alpha': 0.001}

Best Cross-Validation R²: 0.4128


In [25]:
# Evaluating Tuned ElasticNet Regression

elastic_best = elastic_grid.best_estimator_

elastic_r2, elastic_rmse, elastic_mae, elastic_predictions = evaluate_regression_model(
    elastic_best,
    X_test_encoded,
    y_test
)

print("Tuned ElasticNet Regression")
print(f"R²   : {elastic_r2:.4f}")
print(f"RMSE : {elastic_rmse:.4f}")
print(f"MAE  : {elastic_mae:.4f}")

# Save tuned results
elastic_tuned_r2 = elastic_r2
elastic_tuned_rmse = elastic_rmse
elastic_tuned_mae = elastic_mae


Tuned ElasticNet Regression
R²   : 0.4107
RMSE : 5.7999
MAE  : 2.8091


In [26]:
# ElasticNet - Baseline vs Tuned Comparison

elastic_comparison = pd.DataFrame({
    'Model': ['Baseline ElasticNet', 'Tuned ElasticNet'],
    'R²': [elastic_baseline_r2, elastic_tuned_r2],
    'RMSE': [elastic_baseline_rmse, elastic_tuned_rmse],
    'MAE': [elastic_baseline_mae, elastic_tuned_mae]
})

display(elastic_comparison)


,Model,R²,RMSE,MAE
0,Baseline ElasticNet,0.244851,6.565309,3.199979
1,Tuned ElasticNet,0.410670,5.799868,2.809058


In [27]:
# ElasticNet - Coefficients

elastic_coefficients = pd.DataFrame({
    'Feature': all_feature_cols,
    'Coefficient': elastic_best.coef_
}).sort_values(
    by='Coefficient',
    key=np.abs,
    ascending=False
)

print("ElasticNet Regression Coefficients:")
display(elastic_coefficients)


ElasticNet Regression Coefficients:


,Feature,Coefficient
1991,categorical__APR DRG Description_TRACHEOSTOMY ...,9.280099
1380,categorical__APR DRG Code_4,9.280099
91,categorical__Facility Name_Calvary Hospital Inc,7.785402
1605,categorical__APR DRG Code_593,6.360495
1863,categorical__APR DRG Description_NEONATE BIRTH...,6.360495
...,...,...
932,categorical__CCS Procedure Code_11.0,0.000000
934,categorical__CCS Procedure Code_13.0,-0.000000
935,categorical__CCS Procedure Code_14.0,-0.000000
936,categorical__CCS Procedure Code_15.0,-0.000000


In [28]:
# Adding ElasticNet to regression results

regression_results.append({
    'Model': 'ElasticNet Regression',
    'R²': elastic_tuned_r2,
    'RMSE': elastic_tuned_rmse,
    'MAE': elastic_tuned_mae
})

model_registry['ElasticNet Regression'] = {
    'estimator': elastic_best,
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'predictions': elastic_predictions
}


## Polynomial Regression

In [29]:
# Polynomial Regression - Baseline

poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_encoded)
X_test_poly = poly.transform(X_test_encoded)

poly_model = LinearRegression()

poly_model.fit(X_train_poly, y_train)

poly_r2, poly_rmse, poly_mae, poly_predictions = evaluate_regression_model(
    poly_model, 
    X_test_poly,
    y_test
)

print("Polynomial Regression — Baseline (Degree 2)")
print(f"R²   : {poly_r2:.4f}")
print(f"RMSE : {poly_rmse:.4f}")
print(f"MAE  : {poly_mae:.4f}")

# Save baseline results
poly_baseline_r2 = poly_r2
poly_baseline_rmse = poly_rmse
poly_baseline_mae = poly_mae


KeyboardInterrupt: 

In [ ]:
# Polynomial Regression - Hyperparameter Tuning
# Degree 3/4 can create a very large feature matrix. Restricting the
# search to degrees 1 and 2 substantially reduces memory and fit time.

poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('linear', LinearRegression())
])

poly_param_grid = {
    'poly__degree': [1, 2]
}

poly_grid = GridSearchCV(
    estimator=poly_pipeline,
    param_grid=poly_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

poly_grid.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(poly_grid.best_params_)
print(f"\nBest Cross-Validation R²: {poly_grid.best_score_:.4f}")


In [ ]:
# Evaluating Tuned Polynomial Regression

poly_best = poly_grid.best_estimator_

poly_r2, poly_rmse, poly_mae, poly_predictions = evaluate_regression_model(
    poly_best,
    X_test_encoded,
    y_test
)

poly_best_degree = poly_grid.best_params_['poly__degree']

print("Tuned Polynomial Regression")
print(f"Degree: {poly_best_degree}")
print(f"R²   : {poly_r2:.4f}")
print(f"RMSE : {poly_rmse:.4f}")
print(f"MAE  : {poly_mae:.4f}")

# Save tuned results
poly_tuned_r2 = poly_r2
poly_tuned_rmse = poly_rmse
poly_tuned_mae = poly_mae


In [ ]:
# Polynomial Regression - Baseline vs Tuned Comparison

poly_comparison = pd.DataFrame({
    'Model': ['Baseline Polynomial (Degree 2)', f'Tuned Polynomial (Degree {poly_best_degree})'],
    'R²': [poly_baseline_r2, poly_tuned_r2],
    'RMSE': [poly_baseline_rmse, poly_tuned_rmse],
    'MAE': [poly_baseline_mae, poly_tuned_mae]
})

display(poly_comparison)


In [ ]:
# Polynomial Regression - Coefficients

poly_transformer = poly_best.named_steps['poly']
poly_linear = poly_best.named_steps['linear']

poly_feature_names = poly_transformer.get_feature_names_out(all_feature_cols)

poly_coefficients = pd.DataFrame({
    'Feature': poly_feature_names,
    'Coefficient': poly_linear.coef_
}).sort_values(
    by='Coefficient',
    key=np.abs,
    ascending=False
)

print("Polynomial Regression Coefficients:")
display(poly_coefficients.head(20))


In [ ]:
# Adding Polynomial Regression to regression results

regression_results.append({
    'Model': 'Polynomial Regression',
    'R²': poly_tuned_r2,
    'RMSE': poly_tuned_rmse,
    'MAE': poly_tuned_mae
})

model_registry['Polynomial Regression'] = {
    'estimator': poly_best,
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'predictions': poly_predictions
}


## Regressor (6 to 10)

In [ ]:
#Decision Tree Regressor

dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train_encoded, y_train)

dt_r2, dt_rmse, dt_mae, dt_predictions = evaluate_regression_model(
    dt_model,
    X_test_encoded,
    y_test
)

print("Decision Tree Regressor — Baseline")
print(f"R²   : {dt_r2:.4f}")
print(f"RMSE : {dt_rmse:.4f}")
print(f"MAE  : {dt_mae:.4f}")

dt_baseline_r2 = dt_r2
dt_baseline_rmse = dt_rmse
dt_baseline_mae = dt_mae

In [ ]:
# Decision Tree - Hyperparameter Tuning
# RandomizedSearchCV evaluates only a small number of useful candidates.

dt_param_distributions = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=dt_param_distributions,
    n_iter=6,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

dt_grid.fit(X_train, y_train)

print("Best Parameters:")
print(dt_grid.best_params_)
print(f"\nBest Cross-Validation R²: {dt_grid.best_score_:.4f}")


In [ ]:
#Evaluating tuned Decision Tree

dt_best = dt_grid.best_estimator_

dt_r2, dt_rmse, dt_mae, dt_predictions = evaluate_regression_model(
    dt_best,
    X_test,
    y_test
)

print("Tuned Decision Tree Regressor")
print(f"R²   : {dt_r2:.4f}")
print(f"RMSE : {dt_rmse:.4f}")
print(f"MAE  : {dt_mae:.4f}")


dt_tuned_r2 = dt_r2
dt_tuned_rmse = dt_rmse
dt_tuned_mae = dt_mae

In [ ]:
# Decision Tree - Baseline vs Tuned Comparison

dt_comparison = pd.DataFrame({
    'Model': ['Baseline Decision Tree', 'Tuned Decision Tree'],
    'R²': [dt_baseline_r2, dt_tuned_r2],
    'RMSE': [dt_baseline_rmse, dt_tuned_rmse],
    'MAE': [dt_baseline_mae, dt_tuned_mae]
})

display(dt_comparison)

In [ ]:
#Decision Tree - Feature Importance

feature_importance_dt = pd.DataFrame({
    'Feature': all_feature_cols,
    'Importance': dt_best.feature_importances_
}).sort_values(by='Importance',ascending=False)

print("Decision Tree Feature Importance:")
display(feature_importance_dt)

In [ ]:
#Decision Tree - Plot

plt.figure(figsize=(10, 6))

sns.barplot(data=feature_importance_dt,x='Importance',y='Feature')

plt.title("Decision Tree Regressor — Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
#Adding to array to compare every model
regression_results.append({
    'Model': 'Decision Tree Regressor',
    'R²': dt_r2,
    'RMSE': dt_rmse,
    'MAE': dt_mae
})

model_registry['Decision Tree Regressor'] = {
    'estimator': dt_best,
    'X_train': X_train,
    'X_test': X_test,
    'predictions': dt_predictions
}


## Random Forest Regressor

In [ ]:
# Random Forest Regressor - Baseline

rf_model = RandomForestRegressor(random_state=42,n_jobs=-1)

rf_model.fit(X_train, y_train)

rf_r2, rf_rmse, rf_mae, rf_predictions = evaluate_regression_model(
    rf_model,
    X_test,
    y_test
)

print("Random Forest Regressor — Baseline")
print(f"R²   : {rf_r2:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"MAE  : {rf_mae:.4f}")

# Save baseline results
rf_baseline_r2 = rf_r2
rf_baseline_rmse = rf_rmse
rf_baseline_mae = rf_mae

In [ ]:
# Random Forest - Faster Hyperparameter Tuning
# Only two estimator counts are tested and 3-fold CV is used.

rf_param_distributions = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_distributions,
    n_iter=4,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:")
print(rf_grid.best_params_)
print(f"\nBest Cross-Validation R²: {rf_grid.best_score_:.4f}")


In [ ]:
# Evaluating Tuned Random Forest

rf_best = rf_grid.best_estimator_

rf_r2, rf_rmse, rf_mae, rf_predictions = evaluate_regression_model(
    rf_best,
    X_test,
    y_test
)

print("Tuned Random Forest Regressor")
print(f"R²   : {rf_r2:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"MAE  : {rf_mae:.4f}")

# Save tuned results
rf_tuned_r2 = rf_r2
rf_tuned_rmse = rf_rmse
rf_tuned_mae = rf_mae

In [ ]:
# Random Forest - Baseline vs Tuned Comparison

rf_comparison = pd.DataFrame({
    'Model': ['Baseline Random Forest', 'Tuned Random Forest'],
    'R²': [rf_baseline_r2, rf_tuned_r2],
    'RMSE': [rf_baseline_rmse, rf_tuned_rmse],
    'MAE': [rf_baseline_mae, rf_tuned_mae]
})

display(rf_comparison)

In [ ]:
# Random Forest - Feature Importance

feature_importance_rf = pd.DataFrame({
    'Feature': all_feature_cols,
    'Importance': rf_best.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

print("Random Forest Feature Importance:")
display(feature_importance_rf)

In [ ]:
# Random Forest - Feature Importance Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance_rf,
    x='Importance',
    y='Feature'
)

plt.title("Random Forest Regressor — Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Adding Random Forest to regression results

regression_results.append({
    'Model': 'Random Forest Regressor',
    'R²': rf_tuned_r2,
    'RMSE': rf_tuned_rmse,
    'MAE': rf_tuned_mae
})

model_registry['Random Forest Regressor'] = {
    'estimator': rf_best,
    'X_train': X_train,
    'X_test': X_test,
    'predictions': rf_predictions
}


## Histogram Gradient Boosting Regressor


In [ ]:
# Histogram Gradient Boosting Regressor - Baseline

gb_model = HistGradientBoostingRegressor(random_state=42)

gb_model.fit(X_train, y_train)

gb_r2, gb_rmse, gb_mae, gb_predictions = evaluate_regression_model(
    gb_model,
    X_test,
    y_test
)

print("Histogram Gradient Boosting Regressor — Baseline")
print(f"R²   : {gb_r2:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"MAE  : {gb_mae:.4f}")

# Save baseline results
gb_baseline_r2 = gb_r2
gb_baseline_rmse = gb_rmse
gb_baseline_mae = gb_mae


In [ ]:
# Histogram Gradient Boosting - Faster Hyperparameter Tuning
# HistGradientBoosting is substantially faster than the classic
# GradientBoostingRegressor for larger datasets.
# Early stopping lets training stop before all iterations are used.

gb_param_distributions = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_iter': [100, 200],
    'max_leaf_nodes': [15, 31]
}

gb_grid = RandomizedSearchCV(
    estimator=HistGradientBoostingRegressor(
        random_state=42,
        early_stopping=True
    ),
    param_distributions=gb_param_distributions,
    n_iter=4,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

gb_grid.fit(X_train, y_train)

print("Best Parameters:")
print(gb_grid.best_params_)
print(f"\nBest Cross-Validation R²: {gb_grid.best_score_:.4f}")


In [ ]:
# Evaluating Tuned Histogram Gradient Boosting Regressor

gb_best = gb_grid.best_estimator_

gb_r2, gb_rmse, gb_mae, gb_predictions = evaluate_regression_model(
    gb_best,
    X_test,
    y_test
)

print("Tuned Histogram Gradient Boosting Regressor")
print(f"R²   : {gb_r2:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"MAE  : {gb_mae:.4f}")

# Save tuned results
gb_tuned_r2 = gb_r2
gb_tuned_rmse = gb_rmse
gb_tuned_mae = gb_mae


In [ ]:
# Histogram Gradient Boosting - Baseline vs Tuned Comparison

gb_comparison = pd.DataFrame({
    'Model': [
        'Baseline Histogram Gradient Boosting',
        'Tuned Histogram Gradient Boosting'
    ],
    'R²': [
        gb_baseline_r2,
        gb_tuned_r2
    ],
    'RMSE': [
        gb_baseline_rmse,
        gb_tuned_rmse
    ],
    'MAE': [
        gb_baseline_mae,
        gb_tuned_mae
    ]
})

display(gb_comparison)


In [ ]:
# Histogram Gradient Boosting - Feature Importance
# HistGradientBoostingRegressor does not expose feature_importances_,
# so permutation importance is used instead.

gb_permutation = permutation_importance(
    gb_best,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='r2',
    n_jobs=-1
)

feature_importance_gb = pd.DataFrame({
    'Feature': all_feature_cols,
    'Importance': gb_permutation.importances_mean
}).sort_values(
    by='Importance',
    ascending=False
)

print("Histogram Gradient Boosting Feature Importance (Permutation Importance):")
display(feature_importance_gb)


In [ ]:
# Gradient Boosting - Feature Importance Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance_gb,
    x='Importance',
    y='Feature'
)

plt.title("Gradient Boosting Regressor — Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Adding Histogram Gradient Boosting to regression results

regression_results.append({
    'Model': 'Histogram Gradient Boosting Regressor',
    'R²': gb_tuned_r2,
    'RMSE': gb_tuned_rmse,
    'MAE': gb_tuned_mae
})

model_registry['Histogram Gradient Boosting Regressor'] = {
    'estimator': gb_best,
    'X_train': X_train,
    'X_test': X_test,
    'predictions': gb_predictions
}


## Support Vector Regression

In [ ]:
# Support Vector Regression - Baseline

svr_model = SVR()

svr_model.fit(X_train_scaled, y_train)

svr_r2, svr_rmse, svr_mae, svr_predictions = evaluate_regression_model(
    svr_model,
    X_test_scaled,
    y_test
)

print("Support Vector Regression — Baseline")
print(f"R²   : {svr_r2:.4f}")
print(f"RMSE : {svr_rmse:.4f}")
print(f"MAE  : {svr_mae:.4f}")

# Save baseline results
svr_baseline_r2 = svr_r2
svr_baseline_rmse = svr_rmse
svr_baseline_mae = svr_mae

In [ ]:
# SVR - Faster Hyperparameter Tuning
# The linear kernel is intentionally excluded.
# SVR can become very expensive on large datasets, so only a subset
# of the training data is used during hyperparameter search.
# The final best SVR is then refit on the complete training set.

svr_param_distributions = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.01, 0.1, 0.2],
    'kernel': ['rbf']
}

SVR_SEARCH_MAX_SAMPLES = 10000

if len(X_train_scaled) > SVR_SEARCH_MAX_SAMPLES:
    rng = np.random.RandomState(42)
    search_indices = rng.choice(
        len(X_train_scaled),
        size=SVR_SEARCH_MAX_SAMPLES,
        replace=False
    )
    X_svr_search = X_train_scaled[search_indices]
    y_svr_search = y_train.iloc[search_indices] if hasattr(y_train, "iloc") else y_train[search_indices]
else:
    X_svr_search = X_train_scaled
    y_svr_search = y_train

svr_grid = RandomizedSearchCV(
    estimator=SVR(),
    param_distributions=svr_param_distributions,
    n_iter=5,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

svr_grid.fit(X_svr_search, y_svr_search)

print("Best Parameters:")
print(svr_grid.best_params_)
print(f"\nBest Cross-Validation R²: {svr_grid.best_score_:.4f}")
print(f"SVR tuning samples used: {len(X_svr_search):,} / {len(X_train_scaled):,}")

# Refit the selected SVR configuration on the complete training set.
svr_best = SVR(**svr_grid.best_params_)
svr_best.fit(X_train_scaled, y_train)


In [ ]:
# Evaluating Tuned SVR

svr_r2, svr_rmse, svr_mae, svr_predictions = evaluate_regression_model(
    svr_best,
    X_test_scaled,
    y_test
)

print("Tuned SVR")
print(f"R²   : {svr_r2:.4f}")
print(f"RMSE : {svr_rmse:.4f}")
print(f"MAE  : {svr_mae:.4f}")

svr_tuned_r2 = svr_r2
svr_tuned_rmse = svr_rmse
svr_tuned_mae = svr_mae


In [ ]:
# SVR - Baseline vs Tuned Comparison

svr_comparison = pd.DataFrame({
    'Model': [
        'Baseline SVR',
        'Tuned SVR'
    ],
    'R²': [
        svr_baseline_r2,
        svr_tuned_r2
    ],
    'RMSE': [
        svr_baseline_rmse,
        svr_tuned_rmse
    ],
    'MAE': [
        svr_baseline_mae,
        svr_tuned_mae
    ]
})

display(svr_comparison)

In [ ]:
# Adding SVR to regression results

regression_results.append({
    'Model': 'Support Vector Regression',
    'R²': svr_tuned_r2,
    'RMSE': svr_tuned_rmse,
    'MAE': svr_tuned_mae
})

model_registry['Support Vector Regression'] = {
    'estimator': svr_best,
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'predictions': svr_predictions
}


## K-Nearest Neighbors Regressor

In [ ]:
# KNN Regressor - Baseline

knn_model = KNeighborsRegressor()

knn_model.fit(X_train_scaled, y_train)

knn_r2, knn_rmse, knn_mae, knn_predictions = evaluate_regression_model(
    knn_model,
    X_test_scaled,
    y_test
)

print("KNN Regressor — Baseline")
print(f"R²   : {knn_r2:.4f}")
print(f"RMSE : {knn_rmse:.4f}")
print(f"MAE  : {knn_mae:.4f}")

# Save baseline results
knn_baseline_r2 = knn_r2
knn_baseline_rmse = knn_rmse
knn_baseline_mae = knn_mae

In [ ]:
# KNN - Faster Hyperparameter Tuning

knn_param_distributions = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 20],
    'weights': ['uniform', 'distance']
}

knn_grid = RandomizedSearchCV(
    estimator=KNeighborsRegressor(),
    param_distributions=knn_param_distributions,
    n_iter=5,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

knn_grid.fit(X_train_scaled, y_train)

print("Best Parameters:")
print(knn_grid.best_params_)
print(f"\nBest Cross-Validation R²: {knn_grid.best_score_:.4f}")


In [ ]:
# Evaluating Tuned KNN

knn_best = knn_grid.best_estimator_

knn_r2, knn_rmse, knn_mae, knn_predictions = evaluate_regression_model(
    knn_best,
    X_test_scaled,
    y_test
)

print("Tuned KNN Regressor")
print(f"R²   : {knn_r2:.4f}")
print(f"RMSE : {knn_rmse:.4f}")
print(f"MAE  : {knn_mae:.4f}")

# Save tuned results
knn_tuned_r2 = knn_r2
knn_tuned_rmse = knn_rmse
knn_tuned_mae = knn_mae

In [ ]:
# KNN - Baseline vs Tuned Comparison

knn_comparison = pd.DataFrame({
    'Model': [
        'Baseline KNN',
        'Tuned KNN'
    ],
    'R²': [
        knn_baseline_r2,
        knn_tuned_r2
    ],
    'RMSE': [
        knn_baseline_rmse,
        knn_tuned_rmse
    ],
    'MAE': [
        knn_baseline_mae,
        knn_tuned_mae
    ]
})

display(knn_comparison)

In [ ]:
# Adding KNN to regression results

regression_results.append({
    'Model': 'KNN Regressor',
    'R²': knn_tuned_r2,
    'RMSE': knn_tuned_rmse,
    'MAE': knn_tuned_mae
})

model_registry['KNN Regressor'] = {
    'estimator': knn_best,
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'predictions': knn_predictions
}


## Viewing all regression results

In [ ]:
# Final Regression Results

regression_results_df = pd.DataFrame(regression_results)
regression_results_df = regression_results_df.sort_values(
    by='R²', ascending=False
).reset_index(drop=True)

print("All 10 Regression Models — Ranked by R² (Test Set)")
display(regression_results_df)

best_model_name = regression_results_df.iloc[0]['Model']
second_best_model_name = regression_results_df.iloc[1]['Model']

print(f"\nBest performing model : {best_model_name}")
print(f"Runner-up model       : {second_best_model_name}")


## Best Model — Residual & Predicted-vs-Actual Plots

In [ ]:
# Residual plot and Predicted-vs-Actual plot for the best model

best_info = model_registry[best_model_name]
best_y_pred = best_info['predictions']
best_residuals = y_test - best_y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_test, best_y_pred, alpha=0.5, edgecolor='k', linewidth=0.3)
min_val = min(y_test.min(), best_y_pred.min())
max_val = max(y_test.max(), best_y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal fit')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Predicted vs Actual — {best_model_name}')
axes[0].legend()

# Residual plot
axes[1].scatter(best_y_pred, best_residuals, alpha=0.5, edgecolor='k', linewidth=0.3)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Residual Plot — {best_model_name}')

plt.tight_layout()
plt.show()

print(f"Residual mean : {best_residuals.mean():.4f}")
print(f"Residual std  : {best_residuals.std():.4f}")


## 5-Fold Cross-Validated R² — Two Best-Performing Models

In [ ]:
# 3-fold cross-validated R² for the two best-performing models.
# A smaller CV here avoids repeating a costly 5-fold training run.

from sklearn.model_selection import cross_val_score

cv_summary = []

for rank_name in [best_model_name, second_best_model_name]:
    info = model_registry[rank_name]
    cv_scores = cross_val_score(
        info['estimator'],
        info['X_train'],
        y_train,
        cv=3,
        scoring='r2',
        n_jobs=-1
    )
    cv_summary.append({
        'Model': rank_name,
        'CV R² (mean)': cv_scores.mean(),
        'CV R² (std)': cv_scores.std(),
        'CV R² (per fold)': np.round(cv_scores, 4)
    })
    print(f"{rank_name}")
    print(f"  Fold R² scores : {np.round(cv_scores, 4)}")
    print(f"  Mean R²        : {cv_scores.mean():.4f}")
    print(f"  Std  R²        : {cv_scores.std():.4f}\n")

cv_summary_df = pd.DataFrame(cv_summary)
display(cv_summary_df)
